## Step 1: Grabbing the Airbnb data
Using `opendatasets` here so I can pull straight from Kaggle instead of manually downloading zips.

In [2]:
!pip install opendatasets

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import opendatasets as od

In [4]:
od.download("https://www.kaggle.com/datasets/thedevastator/airbnb-prices-in-european-cities")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: fsfgsf
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/thedevastator/airbnb-prices-in-european-cities


100%|██████████| 3.91M/3.91M [00:00<00:00, 42.5MB/s]

### Loading and combining all the city files
The dataset comes split into a bunch of separate CSVs (one per city, and weekday vs weekend). Looping through all of them, tagging each one with whether it's a weekday or weekend price, then stacking everything into one big dataframe.

In [5]:
import glob
import pandas as pd

files = glob.glob("/content/airbnb-prices-in-european-cities/*.csv")

dfs = []

for file in files:
    temp = pd.read_csv(file)


    temp['day_type'] = 'weekday' if 'weekday' in file else 'weekend'

    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)
df.head()

,Unnamed: 0,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,...,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,day_type
0,0,185.799757,Private room,False,True,2.0,True,0,0,10.0,...,1,3.582222,0.174708,105.063613,16.013858,148.940768,31.511339,13.42344,52.49150,weekend
1,1,387.491820,Entire home/apt,False,False,6.0,False,0,1,10.0,...,2,6.082132,0.480956,52.877461,8.059614,66.884920,14.150816,13.50300,52.50900,weekend
2,2,194.914462,Private room,False,True,5.0,False,0,1,9.0,...,1,3.525398,0.511928,75.339762,11.483331,106.443168,22.520139,13.46800,52.51900,weekend
3,3,171.777134,Private room,False,True,2.0,False,0,0,9.0,...,1,3.801739,0.281385,73.668908,11.228659,105.438990,22.307685,13.47096,52.51527,weekend
4,4,207.768533,Private room,False,True,3.0,True,0,0,10.0,...,1,0.982405,0.705579,133.187395,20.300502,198.233241,41.940128,13.42281,52.53139,weekend


### Quick sanity check on the data
Just looking at shapes, dtypes, and whether there are duplicates before touching anything else.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51707 entries, 0 to 51706
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Unnamed: 0                  51707 non-null  int64  
 1   realSum                     51707 non-null  float64
 2   room_type                   51707 non-null  object 
 3   room_shared                 51707 non-null  bool   
 4   room_private                51707 non-null  bool   
 5   person_capacity             51707 non-null  float64
 6   host_is_superhost           51707 non-null  bool   
 7   multi                       51707 non-null  int64  
 8   biz                         51707 non-null  int64  
 9   cleanliness_rating          51707 non-null  float64
 10  guest_satisfaction_overall  51707 non-null  float64
 11  bedrooms                    51707 non-null  int64  
 12  dist                        51707 non-null  float64
 13  metro_dist                  517

In [7]:
df.describe(include='all')

,Unnamed: 0,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,...,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,day_type
count,51707.000000,51707.000000,51707,51707,51707,51707.000000,51707,51707.000000,51707.000000,51707.000000,...,51707.00000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707.000000,51707
unique,NaN,NaN,3,2,2,NaN,2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
top,NaN,NaN,Entire home/apt,False,False,NaN,False,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,weekend
freq,NaN,NaN,32648,51341,33014,NaN,38475,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26207
mean,1620.502388,279.879591,NaN,NaN,NaN,3.161661,NaN,0.291353,0.350204,9.390624,...,1.15876,3.191285,0.681540,294.204105,13.423792,626.856696,22.786177,7.426068,45.671128,NaN
std,1217.380366,327.948386,NaN,NaN,NaN,1.298545,NaN,0.454390,0.477038,0.954868,...,0.62741,2.393803,0.858023,224.754123,9.807985,497.920226,17.804096,9.799725,5.249263,NaN
min,0.000000,34.779339,NaN,NaN,NaN,2.000000,NaN,0.000000,0.000000,2.000000,...,0.00000,0.015045,0.002301,15.152201,0.926301,19.576924,0.592757,-9.226340,37.953000,NaN
25%,646.000000,148.752174,NaN,NaN,NaN,2.000000,NaN,0.000000,0.000000,9.000000,...,1.00000,1.453142,0.248480,136.797385,6.380926,250.854114,8.751480,-0.072500,41.399510,NaN
50%,1334.000000,211.343089,NaN,NaN,NaN,3.000000,NaN,0.000000,0.000000,10.000000,...,1.00000,2.613538,0.413269,234.331748,11.468305,522.052783,17.542238,4.873000,47.506690,NaN
75%,2382.000000,319.694287,NaN,NaN,NaN,4.000000,NaN,1.000000,1.000000,10.000000,...,1.00000,4.263077,0.737840,385.756381,17.415082,832.628988,32.964603,13.518825,51.471885,NaN


In [8]:
df.duplicated().sum()

np.int64(0)

Dropping that stray `Unnamed: 0` column pandas adds when reading the CSVs back in — don't need it.

In [9]:
df.drop(columns=["Unnamed: 0"],inplace=True)

In [10]:
df.head()

,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,day_type
0,185.799757,Private room,False,True,2.0,True,0,0,10.0,98.0,1,3.582222,0.174708,105.063613,16.013858,148.940768,31.511339,13.42344,52.49150,weekend
1,387.491820,Entire home/apt,False,False,6.0,False,0,1,10.0,93.0,2,6.082132,0.480956,52.877461,8.059614,66.884920,14.150816,13.50300,52.50900,weekend
2,194.914462,Private room,False,True,5.0,False,0,1,9.0,86.0,1,3.525398,0.511928,75.339762,11.483331,106.443168,22.520139,13.46800,52.51900,weekend
3,171.777134,Private room,False,True,2.0,False,0,0,9.0,91.0,1,3.801739,0.281385,73.668908,11.228659,105.438990,22.307685,13.47096,52.51527,weekend
4,207.768533,Private room,False,True,3.0,True,0,0,10.0,97.0,1,0.982405,0.705579,133.187395,20.300502,198.233241,41.940128,13.42281,52.53139,weekend


## Step 2: Figuring out city/country from lat & lng
The dataset only gives coordinates, not the actual city or country name, so reverse geocoding each listing to get that info.

In [11]:
!pip install reverse_geocoder

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 23.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for reverse_geocoder: filename=reverse_geocoder-1.5.1-py3-none-any.whl size=2268067 sha256=c02c61f2d3b6bb569bae9854a5ed2dcad65f76db010ec8b82a12b1f8fec92b4e
  Stored in directory: /root/.cache/pip/wheels/11/e1/67/6e47f0ad41ea1843d37e1fbe79c6074744a1f4aace641cf800
Successfully built reverse_geocoder


In [12]:
!pip install pycountry

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 60.1 MB/s eta 0:00:00


In [13]:
import reverse_geocoder as rg
import pycountry
coords = list(zip(df['lat'], df['lng']))
results = rg.search(coords)

df['city'] = [r['name'] for r in results]

df['country'] = [
    pycountry.countries.get(alpha_2=r['cc']).name
    for r in results
]

Loading formatted geocoded file...


In [14]:
results

[{'lat': '52.49376',
  'lon': '13.44469',
  'name': 'Berlin Treptow',
  'admin1': 'Berlin',
  'admin2': '',
  'cc': 'DE'},
 {'lat': '52.51395',
  'lon': '13.49975',
  'name': 'Lichtenberg',
  'admin1': 'Berlin',
  'admin2': '',
  'cc': 'DE'},
 {'lat': '52.52921',
  'lon': '13.47267',
  'name': 'Fennpfuhl',
  'admin1': 'Berlin',
  'admin2': '',
  'cc': 'DE'},
 {'lat': '52.52921',
  'lon': '13.47267',
  'name': 'Fennpfuhl',
  'admin1': 'Berlin',
  'admin2': '',
  'cc': 'DE'},
 {'lat': '52.53878',
  'lon': '13.42443',
  'name': 'Prenzlauer Berg Bezirk',
  'admin1': 'Berlin',
  'admin2': '',
  'cc': 'DE'},
 {'lat': '52.48419',
  'lon': '13.53185',
  'name': 'Karlshorst',
  'admin1': 'Berlin',
  'admin2': '',
  'cc': 'DE'},
 {'lat': '52.51559',
  'lon': '13.45482',
  'name': 'Friedrichshain Bezirk',
  'admin1': 'Berlin',
  'admin2': '',
  'cc': 'DE'},
 {'lat': '52.51559',
  'lon': '13.45482',
  'name': 'Friedrichshain Bezirk',
  'admin1': 'Berlin',
  'admin2': '',
  'cc': 'DE'},
 {'lat': '5

In [15]:
df.head()

,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,...,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,day_type,city,country
0,185.799757,Private room,False,True,2.0,True,0,0,10.0,98.0,...,0.174708,105.063613,16.013858,148.940768,31.511339,13.42344,52.49150,weekend,Berlin Treptow,Germany
1,387.491820,Entire home/apt,False,False,6.0,False,0,1,10.0,93.0,...,0.480956,52.877461,8.059614,66.884920,14.150816,13.50300,52.50900,weekend,Lichtenberg,Germany
2,194.914462,Private room,False,True,5.0,False,0,1,9.0,86.0,...,0.511928,75.339762,11.483331,106.443168,22.520139,13.46800,52.51900,weekend,Fennpfuhl,Germany
3,171.777134,Private room,False,True,2.0,False,0,0,9.0,91.0,...,0.281385,73.668908,11.228659,105.438990,22.307685,13.47096,52.51527,weekend,Fennpfuhl,Germany
4,207.768533,Private room,False,True,3.0,True,0,0,10.0,97.0,...,0.705579,133.187395,20.300502,198.233241,41.940128,13.42281,52.53139,weekend,Prenzlauer Berg Bezirk,Germany


In [16]:
df.tail()

,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,...,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,day_type,city,country
51702,715.938574,Entire home/apt,False,False,6.0,False,0,1,10.0,100.0,...,0.135447,219.402478,15.712158,438.756874,10.604584,16.37940,48.21136,weekend,Vienna,Austria
51703,304.793960,Entire home/apt,False,False,2.0,False,0,0,8.0,86.0,...,0.100839,204.970121,14.678608,342.182813,8.270427,16.38070,48.20296,weekend,Vienna,Austria
51704,637.168969,Entire home/apt,False,False,2.0,False,0,0,10.0,93.0,...,0.202539,169.073402,12.107921,282.296424,6.822996,16.38568,48.20460,weekend,Vienna,Austria
51705,301.054157,Private room,False,True,2.0,False,0,0,10.0,87.0,...,0.287435,109.236574,7.822803,158.563398,3.832416,16.34100,48.19200,weekend,Vienna,Austria
51706,133.230489,Private room,False,True,4.0,True,1,0,10.0,93.0,...,0.480903,150.450381,10.774264,225.247293,5.444140,16.39066,48.20811,weekend,Vienna,Austria


In [17]:
df[df['country'] == 'United Kingdom']

,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,...,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,day_type,city,country
18756,121.122322,Private room,False,True,2.0,False,0,0,6.0,69.0,...,0.437094,222.882243,15.493414,470.088502,8.413765,-0.04975,51.52570,weekend,Bethnal Green,United Kingdom
18757,195.912416,Private room,False,True,2.0,False,1,0,10.0,96.0,...,1.464050,235.385841,16.362588,530.133525,9.488466,-0.08475,51.54210,weekend,Islington,United Kingdom
18758,193.325337,Private room,False,True,3.0,False,1,0,10.0,95.0,...,0.450306,268.913812,18.693247,548.987610,9.825922,-0.14585,51.54802,weekend,Camden Town,United Kingdom
18759,180.389943,Private room,False,True,2.0,False,1,0,9.0,87.0,...,0.132670,472.381314,32.837067,1021.271062,18.278973,-0.10611,51.52108,weekend,Clerkenwell,United Kingdom
18760,405.700981,Entire home/apt,False,False,3.0,False,0,1,7.0,65.0,...,0.354108,318.491470,22.139584,692.775411,12.399473,-0.18797,51.49399,weekend,Kensington,United Kingdom
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43718,310.449446,Private room,False,True,2.0,False,0,0,10.0,100.0,...,0.445862,194.410695,13.513703,446.486289,7.994710,-0.21207,51.48667,weekday,Hammersmith,United Kingdom
43719,265.057974,Entire home/apt,False,False,4.0,False,1,0,8.0,84.0,...,0.463949,254.476513,17.688945,537.720506,9.628334,-0.05459,51.52018,weekday,Bethnal Green,United Kingdom
43720,142.289329,Private room,False,True,2.0,False,0,0,10.0,97.0,...,2.675007,125.891017,8.750824,266.789887,4.777096,-0.12056,51.42875,weekday,Brixton Hill,United Kingdom
43721,372.304146,Private room,False,True,2.0,False,0,0,8.0,80.0,...,1.682697,146.161215,10.159828,325.152018,5.822118,-0.12810,51.44023,weekday,Brixton Hill,United Kingdom


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51707 entries, 0 to 51706
Data columns (total 22 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   realSum                     51707 non-null  float64
 1   room_type                   51707 non-null  object 
 2   room_shared                 51707 non-null  bool   
 3   room_private                51707 non-null  bool   
 4   person_capacity             51707 non-null  float64
 5   host_is_superhost           51707 non-null  bool   
 6   multi                       51707 non-null  int64  
 7   biz                         51707 non-null  int64  
 8   cleanliness_rating          51707 non-null  float64
 9   guest_satisfaction_overall  51707 non-null  float64
 10  bedrooms                    51707 non-null  int64  
 11  dist                        51707 non-null  float64
 12  metro_dist                  51707 non-null  float64
 13  attr_index                  517

In [19]:
df['country'].value_counts()

,count
country,
United Kingdom,9993
France,6688
Portugal,5763
Greece,5280
Holy See (Vatican City State),4586
Italy,4441
Hungary,4022
Austria,3537
Spain,2833


In [20]:
df['city'].value_counts().sort_values(ascending=False).head()

,count
city,
Lisbon,4717
Vatican City,4586
Rome,4383
Vienna,3228
Paris,2718


In [21]:
df['room_type'].unique()

array(['Private room', 'Entire home/apt', 'Shared room'], dtype=object)

## Step 3: Bringing in cost of living data
Pulling a Kaggle dataset that ranks countries by cost of living, rent, groceries, etc. Only need a few of these columns so dropping the rest and renaming to match the other dataframes.

In [22]:
od.download('https://www.kaggle.com/datasets/myrios/cost-of-living-index-by-country-by-number-2024')

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: fgsfg
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/myrios/cost-of-living-index-by-country-by-number-2024


100%|██████████| 2.83k/2.83k [00:00<00:00, 3.36MB/s]

In [23]:
df2=pd.read_csv('/content/cost-of-living-index-by-country-by-number-2024/Cost_of_Living_Index_by_Country_2024.csv')
df2.head()

,Rank,Country,Cost of Living Index,Rent Index,Cost of Living Plus Rent Index,Groceries Index,Restaurant Price Index,Local Purchasing Power Index
0,1,Switzerland,101.1,46.5,74.9,109.1,97.0,158.7
1,2,Bahamas,85.0,36.7,61.8,81.6,83.3,54.6
2,3,Iceland,83.0,39.2,62.0,88.4,86.8,120.3
3,4,Singapore,76.7,67.2,72.1,74.6,50.4,111.1
4,5,Barbados,76.6,19.0,48.9,80.8,69.4,43.5


In [24]:
df2.drop(columns=['Rank','Cost of Living Plus Rent Index','Local Purchasing Power Index'],inplace=True)

In [25]:
df2.head()

,Country,Cost of Living Index,Rent Index,Groceries Index,Restaurant Price Index
0,Switzerland,101.1,46.5,109.1,97.0
1,Bahamas,85.0,36.7,81.6,83.3
2,Iceland,83.0,39.2,88.4,86.8
3,Singapore,76.7,67.2,74.6,50.4
4,Barbados,76.6,19.0,80.8,69.4


In [26]:
df2.rename(columns={'Country':'country'},inplace=True)

## Step 4: Bringing in the World Happiness data
Also grabbing the World Happiness Report so I can eventually compare cost of living against how happy people say they are in each country.

In [27]:
od.download("https://www.kaggle.com/datasets/unsdsn/world-happiness")


Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: fgdg
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/unsdsn/world-happiness


100%|██████████| 36.8k/36.8k [00:00<00:00, 17.7MB/s]

In [28]:
df3 = pd.read_csv('/content/world-happiness/2015.csv')
df3.head()

,Country,Region,Happiness Rank,Happiness Score,Standard Error,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual
0,Switzerland,Western Europe,1,7.587,0.03411,1.39651,1.34951,0.94143,0.66557,0.41978,0.29678,2.51738
1,Iceland,Western Europe,2,7.561,0.04884,1.30232,1.40223,0.94784,0.62877,0.14145,0.43630,2.70201
2,Denmark,Western Europe,3,7.527,0.03328,1.32548,1.36058,0.87464,0.64938,0.48357,0.34139,2.49204
3,Norway,Western Europe,4,7.522,0.03880,1.45900,1.33095,0.88521,0.66973,0.36503,0.34699,2.46531
4,Canada,North America,5,7.427,0.03553,1.32629,1.32261,0.90563,0.63297,0.32957,0.45811,2.45176


In [29]:
df3=df3[['Country','Happiness Score','Economy (GDP per Capita)']]

In [30]:
df3.rename(columns={
    "Country": "country",
    "Happiness Score": "happiness_score",
    "Economy (GDP per Capita)": "gdp_per_capita",
}, inplace=True)

In [31]:
df3.head()

,country,happiness_score,gdp_per_capita
0,Switzerland,7.587,1.39651
1,Iceland,7.561,1.30232
2,Denmark,7.527,1.32548
3,Norway,7.522,1.45900
4,Canada,7.427,1.32629


### Merging cost of living + happiness
Joining these two on `country` so there's one row per country with everything together.

In [32]:
merged_df = pd.merge(df2, df3 , on = 'country',how='inner')
merged_df


,country,Cost of Living Index,Rent Index,Groceries Index,Restaurant Price Index,happiness_score,gdp_per_capita
0,Switzerland,101.1,46.5,109.1,97.0,7.587,1.39651
1,Iceland,83.0,39.2,88.4,86.8,7.561,1.30232
2,Singapore,76.7,67.2,74.6,50.4,6.798,1.52186
3,Norway,76.0,26.2,79.0,73.5,7.522,1.45900
4,Denmark,72.3,26.4,64.8,81.3,7.527,1.32548
...,...,...,...,...,...,...,...
105,Bangladesh,22.5,2.4,25.7,12.8,4.694,0.39753
106,India,21.2,5.6,23.8,15.1,4.565,0.64499
107,Egypt,21.0,3.7,21.2,16.2,4.194,0.88180
108,Libya,20.4,4.3,22.2,15.2,5.754,1.13145


In [33]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   country                 110 non-null    object 
 1   Cost of Living Index    110 non-null    float64
 2   Rent Index              110 non-null    float64
 3   Groceries Index         110 non-null    float64
 4   Restaurant Price Index  110 non-null    float64
 5   happiness_score         110 non-null    float64
 6   gdp_per_capita          110 non-null    float64
dtypes: float64(6), object(1)
memory usage: 6.1+ KB


In [34]:
merged_df['country'].unique()

array(['Switzerland', 'Iceland', 'Singapore', 'Norway', 'Denmark',
       'United States', 'Australia', 'Austria', 'Canada', 'New Zealand',
       'Ireland', 'France', 'Finland', 'Netherlands', 'Israel',
       'Luxembourg', 'Germany', 'United Kingdom', 'Belgium',
       'South Korea', 'Sweden', 'Italy', 'United Arab Emirates', 'Cyprus',
       'Uruguay', 'Jamaica', 'Malta', 'Costa Rica', 'Bahrain', 'Greece',
       'Estonia', 'Qatar', 'Slovenia', 'Latvia', 'Spain', 'Lithuania',
       'Slovakia', 'Czech Republic', 'Panama', 'Japan', 'Croatia',
       'Saudi Arabia', 'Taiwan', 'Portugal', 'Oman', 'Kuwait', 'Albania',
       'Lebanon', 'Hungary', 'Jordan', 'Armenia', 'Poland', 'Mexico',
       'El Salvador', 'Montenegro', 'Chile', 'Guatemala', 'Venezuela',
       'Bulgaria', 'Dominican Republic', 'Serbia', 'Romania', 'Turkey',
       'Cambodia', 'Cameroon', 'Zimbabwe', 'Mauritius', 'Sri Lanka',
       'South Africa', 'Thailand', 'Moldova', 'Georgia', 'Ecuador',
       'Kazakhstan', 'Chi

Quick check that the merge actually worked — average happiness score and GDP per capita across countries.

In [35]:
avg_happiness_score = merged_df['happiness_score'].mean()
avg_gdp_per_capita = merged_df['gdp_per_capita'].mean()
print(f"Average Happiness Score: {avg_happiness_score}")
print(f"Average GDP per Capita: {avg_gdp_per_capita}")



Average Happiness Score: 5.763490909090908
Average GDP per Capita: 1.0024722727272728


In [36]:
avg_happiness_country = merged_df.groupby('country')['happiness_score'].mean().sort_values(ascending=False)
avg_happiness_country

,happiness_score
country,
Switzerland,7.587
Iceland,7.561
Denmark,7.527
Norway,7.522
Canada,7.427
...,...
Uganda,3.931
Cambodia,3.819
Tanzania,3.781


In [37]:
avg_gdp_per_capita_country = merged_df.groupby('country')['gdp_per_capita'].mean().sort_values(ascending=False)
avg_gdp_per_capita_country

,gdp_per_capita
country,
Qatar,1.69042
Luxembourg,1.56391
Kuwait,1.55422
Singapore,1.52186
Norway,1.45900
...,...
Nepal,0.35997
Tanzania,0.28520
Zimbabwe,0.27100


In [38]:
# df = df.copy()

# df.loc[df['city'] == 'Vatican City', 'city'] = 'Rome'

# df.loc[df['country'] == 'Holy See (Vatican City State)', 'country'] = 'Italy'

## Step 5: Merging everything into the main Airbnb dataframe
Now joining the country-level cost-of-living/happiness data onto the actual Airbnb listings.

In [39]:
merged_df1 = pd.merge(df, merged_df, on='country', how='left')
merged_df1

,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,...,lat,day_type,city,country,Cost of Living Index,Rent Index,Groceries Index,Restaurant Price Index,happiness_score,gdp_per_capita
0,185.799757,Private room,False,True,2.0,True,0,0,10.0,98.0,...,52.49150,weekend,Berlin Treptow,Germany,62.2,24.4,60.8,52.8,6.75,1.32792
1,387.491820,Entire home/apt,False,False,6.0,False,0,1,10.0,93.0,...,52.50900,weekend,Lichtenberg,Germany,62.2,24.4,60.8,52.8,6.75,1.32792
2,194.914462,Private room,False,True,5.0,False,0,1,9.0,86.0,...,52.51900,weekend,Fennpfuhl,Germany,62.2,24.4,60.8,52.8,6.75,1.32792
3,171.777134,Private room,False,True,2.0,False,0,0,9.0,91.0,...,52.51527,weekend,Fennpfuhl,Germany,62.2,24.4,60.8,52.8,6.75,1.32792
4,207.768533,Private room,False,True,3.0,True,0,0,10.0,97.0,...,52.53139,weekend,Prenzlauer Berg Bezirk,Germany,62.2,24.4,60.8,52.8,6.75,1.32792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51702,715.938574,Entire home/apt,False,False,6.0,False,0,1,10.0,100.0,...,48.21136,weekend,Vienna,Austria,65.1,22.5,66.4,59.3,7.20,1.33723
51703,304.793960,Entire home/apt,False,False,2.0,False,0,0,8.0,86.0,...,48.20296,weekend,Vienna,Austria,65.1,22.5,66.4,59.3,7.20,1.33723
51704,637.168969,Entire home/apt,False,False,2.0,False,0,0,10.0,93.0,...,48.20460,weekend,Vienna,Austria,65.1,22.5,66.4,59.3,7.20,1.33723
51705,301.054157,Private room,False,True,2.0,False,0,0,10.0,87.0,...,48.19200,weekend,Vienna,Austria,65.1,22.5,66.4,59.3,7.20,1.33723


In [40]:
merged_df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51707 entries, 0 to 51706
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   realSum                     51707 non-null  float64
 1   room_type                   51707 non-null  object 
 2   room_shared                 51707 non-null  bool   
 3   room_private                51707 non-null  bool   
 4   person_capacity             51707 non-null  float64
 5   host_is_superhost           51707 non-null  bool   
 6   multi                       51707 non-null  int64  
 7   biz                         51707 non-null  int64  
 8   cleanliness_rating          51707 non-null  float64
 9   guest_satisfaction_overall  51707 non-null  float64
 10  bedrooms                    51707 non-null  int64  
 11  dist                        51707 non-null  float64
 12  metro_dist                  51707 non-null  float64
 13  attr_index                  517

In [41]:
merged_df1['country'].unique()

array(['Germany', 'France', 'Portugal', 'Hungary', 'Spain', 'Netherlands',
       'Greece', 'United Kingdom', 'Holy See (Vatican City State)',
       'Italy', 'Austria'], dtype=object)

Checking which countries didn't get matched in that merge.

In [42]:
merged_df1[
    merged_df1["happiness_score"].isna()
]["country"].unique()

array(['Holy See (Vatican City State)'], dtype=object)

## Step 6: Filling in Vatican City's missing values
Turns out Vatican City is the only country that didn't match — it's not listed separately in either the cost-of-living or happiness datasets (makes sense, it's basically enclosed within Rome). So instead of dropping it, scraping Italy's numbers as a stand-in using a small Selenium script (`fill_vatican_data.py`) that pulls from Numbeo and Wikipedia and only fills in the cells that are actually missing, without touching anything else.

In [52]:
!apt-get update -q
!pip install -q selenium
!wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -q -y /tmp/chrome.deb

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,083 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,332 kB]
Get:

In [53]:
%%bash
CHROME_VERSION=$(google-chrome-stable --version | grep -oP '[\d]+\.[\d]+\.[\d]+\.[\d]+')
MILESTONE=$(echo $CHROME_VERSION | cut -d. -f1)
echo "Chrome version: $CHROME_VERSION (milestone $MILESTONE)"

DRIVER_URL=$(curl -s https://googlechromelabs.github.io/chrome-for-testing/latest-versions-per-milestone-with-downloads.json \
  | python3 -c "
import json, sys
data = json.load(sys.stdin)
info = data['milestones']['$MILESTONE']
url = [d['url'] for d in info['downloads']['chromedriver'] if d['platform'] == 'linux64'][0]
print(url)
")
echo "Driver URL: $DRIVER_URL"

wget -q -O /tmp/chromedriver.zip "$DRIVER_URL"
unzip -o -q /tmp/chromedriver.zip -d /tmp/chromedriver_extracted
find /tmp/chromedriver_extracted -name chromedriver -exec cp {} /usr/local/bin/chromedriver \;
chmod +x /usr/local/bin/chromedriver
/usr/local/bin/chromedriver --versionmer

Chrome version: 150.0.7871.46 (milestone 150)
Driver URL: https://storage.googleapis.com/chrome-for-testing-public/150.0.7871.46/linux64/chromedriver-linux64.zip
ChromeDriver 150.0.7871.46 (5b586c06e0d27582900f17e2d59c5370d8d6e0bb-refs/branch-heads/7871@{#2039})


In [54]:
from fill_vatican_data import fill_vatican_missing_values
merged_df1 = fill_vatican_missing_values(merged_df1, colab=True)

Scraped values applied: {'Cost of Living Index': 61.3, 'Rent Index': 18.9, 'Groceries Index': 63.8, 'Restaurant Price Index': 64.2}


In [55]:
import importlib
import fill_vatican_data
importlib.reload(fill_vatican_data)
from fill_vatican_data import fill_vatican_missing_values

merged_df1 = fill_vatican_missing_values(merged_df1, colab=True)

Scraped values applied: {'Cost of Living Index': 61.3, 'Rent Index': 18.9, 'Groceries Index': 63.8, 'Restaurant Price Index': 64.2}


Double checking the Vatican City row actually picked up the scraped values this time.

In [56]:
merged_df1.loc[merged_df1["country"] == "Holy See (Vatican City State)"]

,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,...,lat,day_type,city,country,Cost of Living Index,Rent Index,Groceries Index,Restaurant Price Index,happiness_score,gdp_per_capita
24135,156.874664,Private room,False,True,2.0,True,1,0,10.0,95.0,...,41.92498,weekday,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,NaN,NaN
24137,277.745307,Entire home/apt,False,False,4.0,False,0,1,9.0,90.0,...,41.90700,weekday,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,NaN,NaN
24138,444.906834,Entire home/apt,False,False,6.0,False,1,0,9.0,92.0,...,41.90019,weekday,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,NaN,NaN
24142,184.462161,Entire home/apt,False,False,3.0,False,0,0,9.0,92.0,...,41.91250,weekday,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,NaN,NaN
24143,147.522970,Private room,False,True,2.0,True,0,0,9.0,94.0,...,41.90463,weekday,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35870,601.080121,Entire home/apt,False,False,2.0,False,0,1,10.0,92.0,...,41.90636,weekend,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,NaN,NaN
35871,270.497744,Entire home/apt,False,False,4.0,True,1,0,10.0,98.0,...,41.90743,weekend,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,NaN,NaN
35872,413.812452,Entire home/apt,False,False,4.0,False,0,1,10.0,97.0,...,41.90860,weekend,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,NaN,NaN
35873,582.376733,Entire home/apt,False,False,6.0,True,0,1,10.0,96.0,...,41.89100,weekend,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,NaN,NaN


In [57]:
merged_df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51707 entries, 0 to 51706
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   realSum                     51707 non-null  float64
 1   room_type                   51707 non-null  object 
 2   room_shared                 51707 non-null  bool   
 3   room_private                51707 non-null  bool   
 4   person_capacity             51707 non-null  float64
 5   host_is_superhost           51707 non-null  bool   
 6   multi                       51707 non-null  int64  
 7   biz                         51707 non-null  int64  
 8   cleanliness_rating          51707 non-null  float64
 9   guest_satisfaction_overall  51707 non-null  float64
 10  bedrooms                    51707 non-null  int64  
 11  dist                        51707 non-null  float64
 12  metro_dist                  51707 non-null  float64
 13  attr_index                  517

In [58]:
merged_df1 = fill_vatican_missing_values(merged_df1, colab=True)

Scraped values applied: {'Cost of Living Index': 61.3, 'Rent Index': 18.9, 'Groceries Index': 63.8, 'Restaurant Price Index': 64.2}


In [63]:
import sys
for mod in list(sys.modules):
    if "fill_vatican_data" in mod:
        del sys.modules[mod]

import fill_vatican_data
print(fill_vatican_data.__file__)

/content/fill_vatican_data.py


In [64]:
from fill_vatican_data import fill_vatican_missing_values
merged_df1 = fill_vatican_missing_values(merged_df1, colab=True)

[Wikipedia] Found 12 table(s) with class 'wikitable'.
[Wikipedia] Table 0 headers: ['overall rank', 'country or region', 'score', 'log gdp per capita', 'social support', 'healthy life expectancy', 'freedom to make life choices', 'generosity', 'perceptions of corruption', 'dystopia + residual']
Scraped values applied: {'Cost of Living Index': 61.3, 'Rent Index': 18.9, 'Groceries Index': 63.8, 'Restaurant Price Index': 64.2, 'happiness_score': 6.415, 'gdp_per_capita': 1.724}


In [65]:
merged_df1.loc[merged_df1["country"] == "Holy See (Vatican City State)"]

,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,...,lat,day_type,city,country,Cost of Living Index,Rent Index,Groceries Index,Restaurant Price Index,happiness_score,gdp_per_capita
24135,156.874664,Private room,False,True,2.0,True,1,0,10.0,95.0,...,41.92498,weekday,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,6.415,1.724
24137,277.745307,Entire home/apt,False,False,4.0,False,0,1,9.0,90.0,...,41.90700,weekday,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,6.415,1.724
24138,444.906834,Entire home/apt,False,False,6.0,False,1,0,9.0,92.0,...,41.90019,weekday,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,6.415,1.724
24142,184.462161,Entire home/apt,False,False,3.0,False,0,0,9.0,92.0,...,41.91250,weekday,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,6.415,1.724
24143,147.522970,Private room,False,True,2.0,True,0,0,9.0,94.0,...,41.90463,weekday,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,6.415,1.724
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35870,601.080121,Entire home/apt,False,False,2.0,False,0,1,10.0,92.0,...,41.90636,weekend,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,6.415,1.724
35871,270.497744,Entire home/apt,False,False,4.0,True,1,0,10.0,98.0,...,41.90743,weekend,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,6.415,1.724
35872,413.812452,Entire home/apt,False,False,4.0,False,0,1,10.0,97.0,...,41.90860,weekend,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,6.415,1.724
35873,582.376733,Entire home/apt,False,False,6.0,True,0,1,10.0,96.0,...,41.89100,weekend,Vatican City,Holy See (Vatican City State),61.3,18.9,63.8,64.2,6.415,1.724


In [66]:
merged_df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51707 entries, 0 to 51706
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   realSum                     51707 non-null  float64
 1   room_type                   51707 non-null  object 
 2   room_shared                 51707 non-null  bool   
 3   room_private                51707 non-null  bool   
 4   person_capacity             51707 non-null  float64
 5   host_is_superhost           51707 non-null  bool   
 6   multi                       51707 non-null  int64  
 7   biz                         51707 non-null  int64  
 8   cleanliness_rating          51707 non-null  float64
 9   guest_satisfaction_overall  51707 non-null  float64
 10  bedrooms                    51707 non-null  int64  
 11  dist                        51707 non-null  float64
 12  metro_dist                  51707 non-null  float64
 13  attr_index                  517

In [68]:
merged_df1.isnull().sum()

,0
realSum,0
room_type,0
room_shared,0
room_private,0
person_capacity,0
host_is_superhost,0
multi,0
biz,0
cleanliness_rating,0
guest_satisfaction_overall,0


## Done — exporting the final dataset
Saving everything to a CSV and downloading it locally.

In [73]:
from google.colab import files

merged_df1.to_csv("final_df.csv", index=False)

files.download("final_df.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>